# INF0093 - Projeto Prático com Sistemas Multiagentes - 2s 2026
## Prof. Marcelo da Silva Reis
## msreis@unicamp.br

# Aula 6 - Skills, Planejamento e Trace
## Estudo de caso: Assistente de Análise de Editais

Na v3 preliminar, o supervisor decidia um passo de cada vez. Aqui ele escreve o plano inteiro antes de executar, carrega instruções sob demanda e registra a execução em detalhe.

## 1. Configuração e estrutura herdada

Bloco idêntico ao da Aula 5, reunido para o notebook ser autocontido.

In [ ]:
%pip install -q -U langchain langchain-groq langgraph pydantic pandas==2.2.3

In [ ]:
import os, getpass, datetime, platform, time, json, operator, re, unicodedata
from typing import Annotated, Literal, Optional
from typing_extensions import TypedDict
from pydantic import BaseModel, Field

def carregar_chave_groq() -> str:
    if os.environ.get("GROQ_API_KEY"):
        return "variável de ambiente"
    try:
        from google.colab import userdata          # noqa: F401
        os.environ["GROQ_API_KEY"] = userdata.get("MINHA_CHAVE_SECRETA_COLAB")
        return "Colab userdata"
    except Exception:
        os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")
        return "entrada manual"

print("Chave carregada via:", carregar_chave_groq())

from langchain_groq import ChatGroq

MODEL_NAME = "openai/gpt-oss-20b"
TEMPERATURE = 0
llm = ChatGroq(model=MODEL_NAME, temperature=TEMPERATURE)

# Saída estruturada com Groq.
#
# Por padrão, with_structured_output usa tool calling, e o modelo às vezes emite a chamada
# com o nome fora do esperado (por exemplo "plano" em vez de "Plano"). O Groq valida o nome
# de forma estrita e responde 400 com tool_use_failed.
#
# method="json_schema" não passa por tool calling: envia o schema em response_format.
# Com strict=True, gpt-oss-20b e gpt-oss-120b usam decodificação restrita, que garante
# aderência ao schema. Modelos sem suporte caem no fallback para tool calling.
#
def saida_estruturada(schema, include_raw: bool = False):
    try:
        return llm.with_structured_output(schema, method="json_schema", strict=True,
                                          include_raw=include_raw)
    except Exception as erro:
        print(f"json_schema indisponível ({type(erro).__name__}); usando tool calling.")
        return llm.with_structured_output(schema, include_raw=include_raw)


RUN_INFO = {"modelo": MODEL_NAME, "temperatura": TEMPERATURE, "arquitetura": "v4-plano-skills",
            "data": datetime.datetime.now().isoformat(timespec="seconds"),
            "python": platform.python_version()}
RUN_INFO

In [ ]:
call_document = """
CHAMADA PARA PROJETOS DE INOVAÇÃO EM SISTEMAS MULTIAGENTES (AGOSTO DE 2026)

OBJETIVO
Apoiar projetos de inovação tecnológica em sistemas multiagentes, com duração
máxima de 12 meses.

ELEGIBILIDADE
Podem submeter propostas:
- pesquisadores vinculados a universidades brasileiras;
- empresas brasileiras em parceria com uma instituição de pesquisa;
- profissionais com cursos de extensão em sistemas multiagentes.

PRAZO
As propostas devem ser submetidas até 30 de outubro de 2026.

DOCUMENTOS OBRIGATÓRIOS
1. Formulário de submissão;
2. Currículo resumido do coordenador;
3. Plano de trabalho;
4. Orçamento estimado.

RESULTADO
O resultado será divulgado até 15 de dezembro de 2026.
"""

class AnalysisResult(BaseModel):
    answer: str = Field(description="Resposta direta à pergunta.")
    evidence: list[str] = Field(description="Trechos literais do documento.")
    confidence: Literal["high", "medium", "low"]

def normalizar(texto: str) -> str:
    texto = unicodedata.normalize("NFKD", texto.lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", texto).strip()

MARCADORES_AUSENCIA = ["nao esta", "nao consta", "nao foi encontrad", "nao encontrei",
                       "nao informad", "nao especificad", "nao ha informacao", "nao menciona",
                       "nao e mencionad", "ausente no documento", "nao aparece", "nao define",
                       "nao indica"]

def admite_ausencia(r):
    return (any(m in normalizar(r.answer) for m in MARCADORES_AUSENCIA)
            and len(r.evidence) == 0 and r.confidence == "low")

def cobertura_esperada(r, esperado):
    t = normalizar(r.answer)
    return sum(normalizar(k) in t for k in esperado) / len(esperado)

def evidencia_fiel(r, documento):
    if not r.evidence:
        return None
    doc = normalizar(documento)
    return sum(normalizar(e) in doc for e in r.evidence) / len(r.evidence)

def avaliar(caso, r):
    if caso["verificacao"] == "manual":
        return {"aprovado": None, "cobertura": None}
    if caso["esperado"] is None:
        return {"aprovado": admite_ausencia(r), "cobertura": None}
    c = cobertura_esperada(r, caso["esperado"])
    return {"aprovado": c >= caso.get("cobertura_minima", 1.0), "cobertura": round(c, 2)}

test_cases = [
    {"id": "T01", "tipo": "normal", "verificacao": "auto",
     "pergunta": "Qual é o prazo para submissão?",
     "esperado": ["30 de outubro de 2026"], "cobertura_minima": 1.0},
    {"id": "T02", "tipo": "lista", "verificacao": "auto",
     "pergunta": "Quais documentos são obrigatórios?",
     "esperado": ["formulário", "currículo", "plano de trabalho", "orçamento"],
     "cobertura_minima": 1.0},
    {"id": "T03", "tipo": "interpretação", "verificacao": "auto",
     "pergunta": "Quem pode participar?",
     "esperado": ["universidades brasileiras", "empresas brasileiras"], "cobertura_minima": 0.5},
    {"id": "T04", "tipo": "informação ausente", "verificacao": "auto",
     "pergunta": "Qual é o valor máximo de financiamento?", "esperado": None},
    {"id": "T05", "tipo": "ambíguo", "verificacao": "manual",
     "pergunta": "Qual é o prazo?", "esperado": None},
    {"id": "T06", "tipo": "composto", "verificacao": "auto",
     "pergunta": "Quem pode participar e qual é o prazo para submissão?",
     "esperado": ["universidades brasileiras", "30 de outubro de 2026"], "cobertura_minima": 1.0},
    {"id": "T07", "tipo": "composto", "verificacao": "auto",
     "pergunta": "Quem pode participar, quais documentos preciso e até quando envio?",
     "esperado": ["universidades brasileiras", "plano de trabalho", "30 de outubro de 2026"],
     "cobertura_minima": 1.0},
]

print(len(test_cases), "casos")

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

def _secao(documento: str, titulo: str) -> str:
    capturando, coletado = False, []
    for linha in documento.strip().split("\n"):
        if linha.strip().isupper() and len(linha.strip()) > 3:
            if capturando:
                break
            capturando = normalizar(titulo) in normalizar(linha)
            continue
        if capturando and linha.strip():
            coletado.append(linha.strip())
    return "\n".join(coletado)

@tool
def consultar_elegibilidade() -> str:
    """Trecho do edital sobre quem pode submeter propostas."""
    return _secao(call_document, "ELEGIBILIDADE") or "Seção não encontrada."

@tool
def consultar_prazo() -> str:
    """Trecho do edital sobre prazos de submissão."""
    return _secao(call_document, "PRAZO") or "Seção não encontrada."

@tool
def consultar_documentos() -> str:
    """Lista de documentos obrigatórios exigidos pelo edital."""
    return _secao(call_document, "DOCUMENTOS OBRIGATÓRIOS") or "Seção não encontrada."

@tool
def consultar_resultado() -> str:
    """Trecho do edital sobre divulgação de resultados."""
    return _secao(call_document, "RESULTADO") or "Seção não encontrada."

class EstadoAgente(TypedDict):
    messages: Annotated[list, add_messages]

def montar_agente(instrucao: str, tools: list):
    ligado = llm.bind_tools(tools)
    def no_agente(state: EstadoAgente):
        return {"messages": [ligado.invoke([SystemMessage(content=instrucao)] + state["messages"])]}
    b = StateGraph(EstadoAgente)
    b.add_node("agente", no_agente)
    b.add_node("tools", ToolNode(tools))
    b.add_edge(START, "agente")
    b.add_conditional_edges("agente", tools_condition)
    b.add_edge("tools", "agente")
    return b.compile()

def contar(mensagens):
    return {"chamadas_llm": sum(1 for m in mensagens if isinstance(m, AIMessage)),
            "chamadas_tool": sum(len(getattr(m, "tool_calls", None) or []) for m in mensagens),
            "erros_tool": sum(1 for m in mensagens if isinstance(m, ToolMessage)
                              and getattr(m, "status", None) == "error")}

print("[done]")

## 2. Skills

Cada skill reúne o índice (uma linha), a instrução completa e as ferramentas. O índice está sempre
no contexto do planejador; a instrução completa só entra quando a skill é escolhida.

In [ ]:
SKILLS = {
    "consultar_elegibilidade": {
        "resumo": "quem pode submeter propostas e sob que condições",
        "instrucao": (
            "Você analisa critérios de elegibilidade de editais.\n"
            "Consulte a ferramenta disponível antes de responder.\n"
            "Enumere cada perfil elegível separadamente.\n"
            "Cite o trecho literal que sustenta cada perfil.\n"
            "Se a pergunta descrever um perfil não listado, diga que não está previsto, "
            "em vez de aproximá-lo de um perfil parecido.\n"
            "Se a pergunta não for sobre elegibilidade, responda apenas 'fora do escopo'."
        ),
        "tools": [consultar_elegibilidade],
    },
    "consultar_documentos": {
        "resumo": "documentos e anexos obrigatórios da submissão",
        "instrucao": (
            "Você analisa exigências documentais de editais.\n"
            "Consulte a ferramenta disponível antes de responder.\n"
            "Liste todos os itens exigidos, sem omitir nenhum.\n"
            "Cite o trecho literal da lista.\n"
            "Se a pergunta não for sobre documentos, responda apenas 'fora do escopo'."
        ),
        "tools": [consultar_documentos],
    },
    "consultar_prazos": {
        "resumo": "datas de submissão e de divulgação de resultados",
        "instrucao": (
            "Você analisa prazos de editais.\n"
            "Consulte as ferramentas disponíveis antes de responder.\n"
            "Distinga sempre prazo de submissão de data de divulgação de resultado.\n"
            "Cite o trecho literal de cada data.\n"
            "Se a pergunta não for sobre datas, responda apenas 'fora do escopo'."
        ),
        "tools": [consultar_prazo, consultar_resultado],
    }
}

INDICE = "\n".join(f"- {nome}: {s['resumo']}" for nome, s in SKILLS.items())
print(INDICE)

In [ ]:
# Quanto o carregamento sob demanda economiza no contexto do planejador.
#
tudo = sum(len(s["instrucao"]) for s in SKILLS.values())
print(f"índice: {len(INDICE)} caracteres")
print(f"todas as instruções: {tudo} caracteres")
print(f"razão: {tudo / len(INDICE):.1f}x")

Com três skills a diferença é modesta. Ela cresce com o número de skills, e é por isso que o índice
precisa ser bom: a escolha acontece com pouca informação.

## 3. Plano explícito

In [ ]:
class Passo(BaseModel):
    skill: str = Field(description="Nome da skill a executar.")
    objetivo: str = Field(description="O que este passo deve obter.")

class Plano(BaseModel):
    passos: list[Passo] = Field(description="Passos necessários, sem repetir skills.")
    justificativa: str = Field(description="Uma frase sobre a escolha dos passos.")

planejador = saida_estruturada(Plano)

def planejar(pergunta: str, observacoes: str = "") -> Plano:
    prompt = (
        "Monte o plano mínimo para responder à pergunta usando as skills disponíveis.\n"
        "Inclua apenas passos necessários. Se nenhuma skill se aplicar, devolva lista vazia.\n\n"
        f"SKILLS:\n{INDICE}\n\nPERGUNTA: {pergunta}"
    )
    if observacoes:
        prompt += f"\n\nOBSERVAÇÕES DA EXECUÇÃO ANTERIOR:\n{observacoes}"
    return planejador.invoke(prompt)

plano = planejar("Quem pode participar, e até quando envio?")
for p in plano.passos:
    print("-", p.skill, ":", p.objetivo)
print("\nJustificativa:", plano.justificativa)

O plano é um artefato: dá para lê-lo, avaliá-lo e compará-lo com o que aconteceu de fato.

## 4. Execução com trace

O executor monta o agente da skill no momento em que ela é usada, e registra cada passo.
Quando um passo devolve algo fora do escopo ou vazio, o sistema replaneja uma única vez.

In [ ]:
SYSTEM_FORMAT = """
Converta a análise abaixo no formato estruturado.
'evidence' deve conter apenas trechos copiados literalmente do documento.
Se a informação não constar, diga isso na 'answer', deixe 'evidence' vazia e use 'confidence' baixa.
"""

estruturado = saida_estruturada(AnalysisResult, include_raw=True)

class EstadoV3e(TypedDict):
    pergunta: str
    plano: list
    pendentes: list
    achados: Annotated[list, operator.add]
    trace: Annotated[list, operator.add]
    replanejou: bool
    resultado: Optional[AnalysisResult]

def no_planejador(state: EstadoV3e):
    inicio = time.perf_counter()
    p = planejar(state["pergunta"])
    passos = [{"skill": x.skill, "objetivo": x.objetivo} for x in p.passos if x.skill in SKILLS]
    return {"plano": passos, "pendentes": list(passos),
            "trace": [{"etapa": "planejador", "plano": [x["skill"] for x in passos],
                       "justificativa": p.justificativa, "chamadas_llm": 1, "chamadas_tool": 0,
                       "latencia_s": round(time.perf_counter() - inicio, 2)}]}

def no_executor(state: EstadoV3e):
    passo = state["pendentes"][0]
    restantes = state["pendentes"][1:]
    skill = SKILLS[passo["skill"]]

    inicio = time.perf_counter()
    agente = montar_agente(skill["instrucao"], skill["tools"])
    estado = agente.invoke(
        {"messages": [HumanMessage(content=f"{state['pergunta']}\n\nFoco: {passo['objetivo']}")]},
        config={"recursion_limit": 8},
    )
    m = contar(estado["messages"])
    conteudo = str(estado["messages"][-1].content)
    util = "fora do escopo" not in normalizar(conteudo) and len(conteudo.strip()) > 0

    return {"pendentes": restantes,
            "achados": [{"skill": passo["skill"], "conteudo": conteudo, "util": util}],
            "trace": [{"etapa": f"skill:{passo['skill']}", "objetivo": passo["objetivo"],
                       "util": util, "latencia_s": round(time.perf_counter() - inicio, 2), **m}]}

def no_replanejador(state: EstadoV3e):
    inicio = time.perf_counter()
    falhas = [a["skill"] for a in state["achados"] if not a["util"]]
    obs = f"As skills {falhas} não trouxeram informação útil."
    p = planejar(state["pergunta"], obs)
    ja = {a["skill"] for a in state["achados"] if a["util"]}
    novos = [{"skill": x.skill, "objetivo": x.objetivo}
             for x in p.passos if x.skill in SKILLS and x.skill not in ja]
    return {"pendentes": novos, "replanejou": True,
            "trace": [{"etapa": "replanejador", "motivo": obs,
                       "plano": [x["skill"] for x in novos], "chamadas_llm": 1,
                       "chamadas_tool": 0, "latencia_s": round(time.perf_counter() - inicio, 2)}]}

def no_sintetizador(state: EstadoV3e):
    inicio = time.perf_counter()
    material = "\n\n".join(f"[{a['skill']}]\n{a['conteudo']}"
                           for a in state["achados"] if a["util"])
    saida = estruturado.invoke(
        SYSTEM_FORMAT + "\n\nDOCUMENTO:\n" + call_document
        + "\n\nPERGUNTA:\n" + state["pergunta"]
        + "\n\nANÁLISE:\n" + (material or "(nenhum achado útil)")
    )
    return {"resultado": saida["parsed"],
            "trace": [{"etapa": "sintetizador", "achados_usados": material.count("["),
                       "chamadas_llm": 1, "chamadas_tool": 0,
                       "latencia_s": round(time.perf_counter() - inicio, 2)}]}

def rotear(state: EstadoV3e):
    if state["pendentes"]:
        return "executor"
    houve_falha = any(not a["util"] for a in state["achados"])
    if houve_falha and not state["replanejou"]:
        return "replanejador"
    return "sintetizador"

print("[done]")

In [ ]:
b = StateGraph(EstadoV3e)

b.add_node("planejador", no_planejador)
b.add_node("executor", no_executor)
b.add_node("replanejador", no_replanejador)
b.add_node("sintetizador", no_sintetizador)

b.add_edge(START, "planejador")
b.add_conditional_edges("planejador", rotear,
                        {"executor": "executor", "replanejador": "replanejador",
                         "sintetizador": "sintetizador"})
b.add_conditional_edges("executor", rotear,
                        {"executor": "executor", "replanejador": "replanejador",
                         "sintetizador": "sintetizador"})
b.add_conditional_edges("replanejador", rotear,
                        {"executor": "executor", "sintetizador": "sintetizador",
                         "replanejador": "sintetizador"})

b.add_edge("sintetizador", END)

app_v3e = b.compile()

def responder_v3e(pergunta: str, limite: int = 25):
    inicio = time.perf_counter()
    estado = app_v3e.invoke(
        {"pergunta": pergunta, "plano": [], "pendentes": [], "achados": [], "trace": [],
         "replanejou": False, "resultado": None},
        config={"recursion_limit": limite},
    )
    trace = estado["trace"]
    metricas = {
        "chamadas_llm": sum(t.get("chamadas_llm", 0) for t in trace),
        "chamadas_tool": sum(t.get("chamadas_tool", 0) for t in trace),
        "erros_tool": sum(t.get("erros_tool", 0) for t in trace),
        "passos": [t["etapa"] for t in trace if t["etapa"].startswith("skill:")],
        "replanejou": estado["replanejou"],
        "latencia_s": round(time.perf_counter() - inicio, 2),
    }
    return estado["resultado"], metricas, trace

print("[v3e pronta]")

In [ ]:
# Visualização do grafo do v3 estendido (opcional, requer IPython).
#
try:
    from IPython.display import Image, display
    print("Nosso v3 estendido:")
    display(Image(app_v3e.get_graph().draw_mermaid_png()))
except Exception as erro:
    print("Sem renderização de imagem:", type(erro).__name__)
    print(app_v3e.get_graph().draw_ascii())


In [ ]:
resultado, metricas, trace = responder_v3e(
    "Quem pode participar, quais documentos preciso e até quando envio?")

print("RESPOSTA:", resultado.answer)
print("MÉTRICAS:", metricas)
print()
for t in trace:
    print(" ", t["etapa"], "|", {k: v for k, v in t.items() if k != "etapa"})

## 5. Atribuição de custo

In [ ]:
import pandas as pd

def custo_por_etapa(trace):
    df = pd.DataFrame(trace).fillna(0)
    colunas = [c for c in ["latencia_s", "chamadas_llm", "chamadas_tool"] if c in df]
    resumo = df.groupby("etapa")[colunas].sum()
    if "latencia_s" in resumo:
        total = resumo["latencia_s"].sum()
        resumo["% latência"] = (100 * resumo["latencia_s"] / total).round(1) if total else 0
    return resumo.sort_values("latencia_s", ascending=False)

custo_por_etapa(trace)

Uma etapa que concentra a maior parte da latência é candidata a modelo menor, cache ou remoção.
Sem essa tabela, a decisão seria por intuição.

## 6. Usando o trace para depurar

A cadeia tem quatro elos. O trace diz em qual deles a informação se perdeu.

In [ ]:
def diagnosticar(pergunta: str, termo: str):
    """Segue um termo esperado da fonte até a resposta final."""
    resultado, metricas, trace = responder_v3e(pergunta)
    alvo = normalizar(termo)

    na_fonte = alvo in normalizar(call_document)
    passos = [t for t in trace if t["etapa"].startswith("skill:")]
    achados = [t for t in passos if t.get("util")]
    na_resposta = alvo in normalizar(resultado.answer)

    print(f"termo: {termo!r}")
    print(" 1. está no documento          :", na_fonte)
    print(" 2. alguma skill foi acionada  :", [t['etapa'] for t in passos] or "nenhuma")
    print(" 3. skills com achado útil     :", [t['etapa'] for t in achados] or "nenhuma")
    print(" 4. aparece na resposta final  :", na_resposta)
    if na_fonte and not na_resposta:
        if not passos:
            print(" -> o plano não previu a skill necessária")
        elif not achados:
            print(" -> a skill foi acionada mas não trouxe o dado")
        else:
            print(" -> o dado foi coletado e perdido na síntese")
    return resultado

_ = diagnosticar("Quem pode participar, quais documentos preciso e até quando envio?",
                 "plano de trabalho")

## 7. Comparação v3 e v3e

A v3 da Aula 5 é reconstruída aqui em versão mínima, com supervisor passo a passo, para que as duas
sejam medidas na mesma execução.

In [ ]:
INSTRUCAO_V3 = """
Você é um assistente de análise de editais.
Use as ferramentas para obter os trechos necessários.
Para perguntas compostas, consulte TODAS as informações necessárias antes de responder.
Não invente informações.
"""

agente_v3 = montar_agente(INSTRUCAO_V3,
                          [consultar_elegibilidade, consultar_prazo,
                           consultar_documentos, consultar_resultado])

def responder_v3(pergunta: str, limite: int = 12):
    inicio = time.perf_counter()
    estado = agente_v3.invoke({"messages": [HumanMessage(content=pergunta)]},
                              config={"recursion_limit": limite})
    m = contar(estado["messages"])
    material = "\n".join(str(getattr(x, "content", "")) for x in estado["messages"])
    saida = estruturado.invoke(
        SYSTEM_FORMAT + "\n\nDOCUMENTO:\n" + call_document
        + "\n\nPERGUNTA:\n" + pergunta + "\n\nANÁLISE:\n" + material)
    m["chamadas_llm"] += 1
    m["latencia_s"] = round(time.perf_counter() - inicio, 2)
    return saida["parsed"], m

registros = []
for caso in test_cases:
    r3, m3 = responder_v3(caso["pergunta"])
    r3e, m3e, t3e = responder_v3e(caso["pergunta"])
    a3, a3e = avaliar(caso, r3), avaliar(caso, r3e)
    registros.append({
        "id": caso["id"], "tipo": caso["tipo"],
        "v3_aprovado": a3["aprovado"], "v3e_aprovado": a3e["aprovado"],
        "v3_latencia": m3["latencia_s"], "v3e_latencia": m3e["latencia_s"],
        "v3_llm": m3["chamadas_llm"], "v3e_llm": m3e["chamadas_llm"],
        "plano": " > ".join(p.replace("skill:", "") for p in m3e["passos"]) or "(vazio)",
        "replanejou": m3e["replanejou"],
    })
    print(f'[{caso["id"]}] v3={a3["aprovado"]} v3e={a3e["aprovado"]} | plano: {registros[-1]["plano"]}')

df = pd.DataFrame(registros)
df

In [ ]:
autos = df[df["v3_aprovado"].notna()]
COMPARACAO = {
    "casos_automaticos": int(len(autos)),
    "v3_taxa": round(float(autos["v3_aprovado"].astype(bool).mean()), 2),
    "v3e_taxa": round(float(autos["v3e_aprovado"].astype(bool).mean()), 2),
    "v3_latencia_mediana_s": round(float(df["v3_latencia"].median()), 2),
    "v3e_latencia_mediana_s": round(float(df["v3e_latencia"].median()), 2),
    "v3_chamadas_llm": int(df["v3_llm"].sum()),
    "v3e_chamadas_llm": int(df["v3e_llm"].sum()),
    "replanejamentos": int(df["replanejou"].sum()),
}

with open("v3e_vs_v3_resultados.json", "w", encoding="utf-8") as f:
    json.dump({"run": RUN_INFO, "comparacao": COMPARACAO, "registros": registros},
              f, ensure_ascii=False, indent=2, default=str)

COMPARACAO

## 8. Discussão

1. O plano previu os passos certos, ou pediu skills desnecessárias?
2. Houve replanejamento? O que o disparou?
3. O plano explícito melhorou a taxa de acerto, ou apenas deixou a execução mais legível?
4. Qual etapa domina a latência, e o que você faria com essa informação?
5. Em que caso o trace explicou uma falha que o resultado sozinho não explicava?

## 9. Exercício

1. Escreva o índice das skills do seu sistema, uma linha por skill.
2. Separe instrução completa do índice e carregue sob demanda.
3. Decida se o seu problema pede plano explícito, e justifique pela natureza da tarefa.
4. Registre trace suficiente para responder às três perguntas da aula.
5. Produza a tabela de custo por etapa e comente o que ela revelou.